# 📊 Loan Default Prediction — Exploratory Data Analysis

**Objective:** Understand the dataset, identify patterns, handle data quality issues,
and engineer features before model training.

## Table of Contents
1. [Data Loading](#1-data-loading)
2. [Dataset Overview](#2-dataset-overview)
3. [Missing Value Analysis](#3-missing-value-analysis)
4. [Target Variable Analysis](#4-target-variable-analysis)
5. [Numerical Feature Analysis](#5-numerical-feature-analysis)
6. [Categorical Feature Analysis](#6-categorical-feature-analysis)
7. [Correlation Analysis](#7-correlation-analysis)
8. [Outlier Detection](#8-outlier-detection)
9. [Feature Engineering](#9-feature-engineering)
10. [Key Insights & Conclusion](#10-key-insights--conclusion)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
sns.set_palette('Set2')
sns.set_style('whitegrid')

print('Libraries loaded ✅')

## 1. Data Loading

In [ ]:
# Load dataset
df = pd.read_csv('data/loan_data.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 2. Dataset Overview

In [ ]:
print('=' * 60)
print('DATASET INFO')
print('=' * 60)
df.info()

print('\n' + '=' * 60)
print('STATISTICAL SUMMARY')
print('=' * 60)
df.describe(include='all').round(2)

In [ ]:
# Data types breakdown
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

print(f'Numeric columns ({len(numeric_cols)}): {numeric_cols}')
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')

## 3. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing_df.empty:
    print('✅ No missing values found!')
else:
    print(missing_df)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_df['Missing %'].plot(kind='bar', ax=ax, color='salmon')
    ax.set_title('Missing Values by Feature (%)')
    ax.set_ylabel('Missing %')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 4. Target Variable Analysis

In [ ]:
# Identify target column
target_col = None
for candidate in ['Default', 'loan_status', 'Loan_Status', 'default', 'TARGET']:
    if candidate in df.columns:
        target_col = candidate
        break

print(f'Target column: {target_col}')
print(f'\nValue counts:')
print(df[target_col].value_counts())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df[target_col].value_counts()
ax1.bar(counts.index.astype(str), counts.values, color=['#2ecc71', '#e74c3c'])
ax1.set_title(f'Target Variable Distribution: {target_col}')
ax1.set_xlabel('Class')
ax1.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax1.text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
ax2.pie(counts.values, labels=counts.index.astype(str),
        autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
        startangle=90, explode=[0, 0.05])
ax2.set_title('Class Imbalance Ratio')

plt.tight_layout()
plt.show()

imbalance_ratio = counts.max() / counts.min()
print(f'\nImbalance ratio: {imbalance_ratio:.2f}x (will handle with SMOTE)')

## 5. Numerical Feature Analysis

In [ ]:
plot_cols = [c for c in numeric_cols if c != target_col][:8]  # first 8

n_cols = 2
n_rows = (len(plot_cols) + 1) // 2

fig, axes = plt.subplots(n_rows, n_cols * 2, figsize=(20, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    # Histogram
    ax_hist = axes[i * 2]
    df[col].hist(ax=ax_hist, bins=30, color='steelblue', edgecolor='white')
    ax_hist.set_title(f'{col} — Distribution')
    ax_hist.set_xlabel(col)
    
    # Box plot by target
    ax_box = axes[i * 2 + 1]
    try:
        df.boxplot(column=col, by=target_col, ax=ax_box)
        ax_box.set_title(f'{col} by {target_col}')
        ax_box.set_xlabel(target_col)
    except:
        ax_box.set_visible(False)

# Hide unused axes
for j in range(len(plot_cols) * 2, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 6. Categorical Feature Analysis

In [ ]:
if cat_cols:
    fig, axes = plt.subplots(1, len(cat_cols), figsize=(6 * len(cat_cols), 5))
    if len(cat_cols) == 1:
        axes = [axes]
    
    for ax, col in zip(axes, cat_cols):
        if col == target_col:
            continue
        try:
            ct = pd.crosstab(df[col], df[target_col], normalize='index') * 100
            ct.plot(kind='bar', ax=ax, stacked=True, colormap='RdYlGn_r')
            ax.set_title(f'Default Rate by {col}')
            ax.set_ylabel('Percentage (%)')
            ax.set_xlabel(col)
            ax.legend(title=target_col, loc='upper right')
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
        except Exception as e:
            ax.text(0.5, 0.5, f'Error: {e}', ha='center', va='center', transform=ax.transAxes)
    
    plt.tight_layout()
    plt.show()
else:
    print('No categorical features found.')

## 7. Correlation Analysis

In [ ]:
# Encode target for correlation
df_corr = df.copy()
if df_corr[target_col].dtype == object:
    df_corr[target_col] = df_corr[target_col].map({'Y': 1, 'N': 0, 'Yes': 1, 'No': 0,
                                                    'Approved': 0, 'Rejected': 1})

corr_matrix = df_corr[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, ax=ax,
    linewidths=0.5, annot_kws={'size': 9}
)
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

# Correlation with target
if target_col in corr_matrix.columns:
    target_corr = corr_matrix[target_col].drop(target_col).sort_values(key=abs, ascending=False)
    print('\nCorrelation with Target:')
    print(target_corr.round(3))

## 8. Outlier Detection

In [ ]:
print('Outlier Analysis (IQR method):')
print('-' * 50)

outlier_summary = []
for col in [c for c in numeric_cols if c != target_col]:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = round(n_outliers / len(df) * 100, 2)
    outlier_summary.append({'Feature': col, 'Outliers': n_outliers, 'Outlier %': pct,
                             'Lower Fence': round(lower, 2), 'Upper Fence': round(upper, 2)})

outlier_df = pd.DataFrame(outlier_summary).sort_values('Outlier %', ascending=False)
print(outlier_df.to_string(index=False))

In [ ]:
# Box plots for outlier visualization
top_outlier_cols = outlier_df[outlier_df['Outlier %'] > 0]['Feature'].tolist()[:6]

if top_outlier_cols:
    fig, axes = plt.subplots(1, len(top_outlier_cols), figsize=(4 * len(top_outlier_cols), 5))
    if len(top_outlier_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, top_outlier_cols):
        ax.boxplot(df[col].dropna(), patch_artist=True,
                   boxprops=dict(facecolor='lightblue'))
        ax.set_title(col)
    plt.suptitle('Outlier Visualization (Box Plots)', fontsize=13)
    plt.tight_layout()
    plt.show()

## 9. Feature Engineering

In [ ]:
df_eng = df.copy()

new_features = []

if 'income_annum' in df_eng.columns and 'loan_amount' in df_eng.columns:
    df_eng['loan_to_income'] = df_eng['loan_amount'] / (df_eng['income_annum'] + 1)
    new_features.append('loan_to_income')
    print('✅ Created: loan_to_income')

if 'loan_amount' in df_eng.columns and 'loan_term' in df_eng.columns:
    df_eng['monthly_payment_est'] = df_eng['loan_amount'] / (df_eng['loan_term'] + 1)
    new_features.append('monthly_payment_est')
    print('✅ Created: monthly_payment_est')

if 'cibil_score' in df_eng.columns:
    df_eng['good_credit'] = (df_eng['cibil_score'] >= 700).astype(int)
    new_features.append('good_credit')
    print('✅ Created: good_credit (1 if CIBIL ≥ 700)')

if 'no_of_dependents' in df_eng.columns and 'income_annum' in df_eng.columns:
    df_eng['income_per_dependent'] = df_eng['income_annum'] / (df_eng['no_of_dependents'] + 1)
    new_features.append('income_per_dependent')
    print('✅ Created: income_per_dependent')

total_asset_cols = [c for c in df_eng.columns if 'asset' in c.lower()]
if total_asset_cols:
    df_eng['total_assets'] = df_eng[total_asset_cols].sum(axis=1)
    new_features.append('total_assets')
    print(f'✅ Created: total_assets (sum of {total_asset_cols})')

print(f'\nNew features created: {new_features}')

In [ ]:
# Visualize new engineered features vs target
if new_features and target_col:
    df_plot = df_eng.copy()
    if df_plot[target_col].dtype == object:
        df_plot[target_col] = df_plot[target_col].map({'Y': 1, 'N': 0, 'Yes': 1, 'No': 0,
                                                        'Approved': 0, 'Rejected': 1})
    
    n = len(new_features)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    
    for ax, feat in zip(axes, new_features):
        for cls in df_plot[target_col].dropna().unique():
            subset = df_plot[df_plot[target_col] == cls][feat].dropna()
            ax.hist(subset, bins=30, alpha=0.6, label=f'Class {cls}')
        ax.set_title(f'{feat} by Target')
        ax.legend()
    
    plt.suptitle('Engineered Features vs Target', fontsize=13)
    plt.tight_layout()
    plt.show()

## 10. Key Insights & Conclusion

In [ ]:
print('=' * 70)
print('KEY EDA INSIGHTS — Loan Default Prediction')
print('=' * 70)

# Class imbalance
vc = df[target_col].value_counts()
print(f'\n📊 Class Imbalance:')
print(f'   Majority class: {vc.index[0]} → {vc.iloc[0]} ({vc.iloc[0]/len(df)*100:.1f}%)')
print(f'   Minority class: {vc.index[-1]} → {vc.iloc[-1]} ({vc.iloc[-1]/len(df)*100:.1f}%)')
print(f'   Strategy: SMOTE oversampling in model training pipeline')

# Missing values
total_missing = df.isnull().sum().sum()
print(f'\n🔍 Missing Values: {total_missing} total')
if total_missing > 0:
    print(f'   Strategy: Median imputation (numeric), Mode imputation (categorical)')

# Outliers
print(f'\n⚠️  Outliers: Detected via IQR × 1.5')
print(f'   Strategy: Clip to [Q1 - 1.5×IQR, Q3 + 1.5×IQR]')

# Feature engineering summary
print(f'\n🔧 Feature Engineering:')
print(f'   - loan_to_income: financial stress indicator')
print(f'   - monthly_payment_est: affordability proxy')
print(f'   - good_credit: CIBIL threshold flag')
print(f'   - income_per_dependent: family financial burden')
print(f'   - total_assets: overall collateral strength')

print(f'\n✅ EDA Complete. Proceed to model training (model.py).')